In [1]:
import numpy as np
import json
import faiss
from sentence_transformers import SentenceTransformer

In [2]:
NUM_CHAPTERS = 10
with open(f'first{NUM_CHAPTERS}_chunks.json', 'r', encoding='utf-8') as file:
    chunks = json.load(file)

In [3]:
class FaissRetriever:
    def __init__(self, model_name='BAAI/bge-small-zh-v1.5', chunk_file=f'first{NUM_CHAPTERS}_chunks.json'):
        self.model = SentenceTransformer(model_name)
        self.index = None
        with open(chunk_file, 'r', encoding='utf-8') as file:
            self.chunks = json.load(file)
        # 用模型嵌入之后转成 np.array
        embeddings = self.model.encode([chunk['chunk_text'] for chunk in self.chunks], show_progress_bar=True)
        embeddings = np.array(embeddings).astype('float32')
        dimension = embeddings.shape[1]
        # 建立 FAISS 索引，FlatL2 是暴力搜索
        self.index = faiss.IndexFlatL2(dimension)
        self.index.add(embeddings)
        print(f"FAISS 添加了 {self.index.ntotal} 个 {dimension} 维向量。")
    def retrieve(self, query, top_k=5):
        query_embedding = np.array(self.model.encode([query])).astype('float32')
        # distances 是基于 FlatL2 的距离，indices 是对应的向量索引
        distances, indices = self.index.search(query_embedding, top_k)
        results = []
        for idx in indices[0]:
            results.append({
                'distance': float(distances[0][list(indices[0]).index(idx)]),
                'content': self.chunks[idx]
            })
        return results

In [ ]:
# retriever = FaissRetriever()
from vdb import FaissRetriever
retriever = FaissRetriever(chunks_file=f'first{NUM_CHAPTERS}_chunks.json')

已加载 FAISS 索引，包含 62 个向量。


In [5]:
query = "杨奇修炼到了什么境界？"
results = retriever.retrieve(query, top_k=5)
for res in results:
    print(f"Chapter ID: {res['chpt_id']}, Chunk ID: {res['chunk_id']}, Score: {res['score']}, Content: {res['text_snippet']}\n")

Chapter ID: 5, Chunk ID: 1, Score: 0.6432148218154907, Content: 神象一般的力量，地狱一般的深邃。
这门神功实在是太可怕。
在这三天的修炼中，杨奇气功掌握比以前熟练了许多，而且能够运用这门神功，完美的掩饰自己的气息，就算是高手用气功深入他的经脉查探，也仍旧会以为他是一个废物。
他经历了这一场大变故，深深知道，藏拙的重要性。
再这样修炼下去，他会很快的进入气功六段，兵气的境界。凝气成兵，聚成实质，手发剑气，一剑迸射，洞穿铁甲。
六段“兵气”的境界，比起五段“暴气”，又强了许多倍，杀伤力根本不是暴气能够比拟的。有的五段暴气高手，终其一生，也无法进入“兵气”的境界，就算是天资过人的强者，也起码要十年以上的苦修，才能够凝气成兵。
杨奇虽然是青年才俊，但是还没有见识过兵气境高手的厉害。
这三天的时间，杨奇也尝试着用意念沟通自己眉心的那个金色小人，但是金色小人一动不动，如同神灵，根本不理会他。
这金色小人，虽然在他的身体之中，但似乎是处于了一个异度空间之内，只有他的意念能够沟通，就算是再高强的人，也无法看到。
“到底是何方神圣？寄居在自己眉心内，它能够传授给自己神象镇狱劲这等无上气功，想必是个活物？为什么不理我？难道我的境界，现在还不足以和他沟通？”杨奇心中很是疑惑。
不过这金色小人不理会他，他也没有办法奈何，只能够望洋兴叹。
端坐在自己房屋中，他再次进入了深沉的冥想，揣摩神象镇狱劲的奥秘……
第三天一大早，天刚蒙蒙亮。
整个燕都城的八个城门口，就出现了许多马车，一支支的商队，都赶向了燕都城中的杨家府邸，这让许多人都为之侧目。
“出了什么事？怎么这么多的商队都到杨家去了？这些人是什么人？”

Chapter ID: 5, Chunk ID: 0, Score: 0.6503486633300781, Content: 一连三天，杨奇都悄悄的溜出去，晚上修炼“神象镇狱劲”，而白天他则是装作了一幅养伤的模样，深藏不露，让所有的人都觉得自己功力被废，无法恢复。
毕竟，功力被废除了的人，基本上不能够恢复。
如果他突然恢复，而且还突飞猛进，肯定要被许多人注意到，起码城主府的高手会关注自己，不是一个好事情，万一“神象镇狱劲”被暴露，恐怕整个杨家都会被夷为平地。
他越修炼，越感觉到这门功法的博大精深，不可思议，真是盖世奇功。
要暴